## 医疗分诊 LoRA 模型评估.ipynb

### Notebook 功能说明
专门用于 DistilBERT-LoRA 医疗四分类分诊模型离线评估
输出内容：
- 每类别精准率 / 召回率 / F1
- 整体分类报告
- 混淆矩阵（分析模型误诊偏向）
评估数据集：完全未见过的疑问句句式（真实泛化能力）

### 1. 导入依赖 & 全局常量

In [1]:
import env_config

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import torch
from peft import PeftModel
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 全局配置（与训练脚本完全对齐）
LABELS = ["emergency", "urgent", "routine", "self_care"]
BASE_MODEL = "distilbert-base-uncased"
OUT_DIR = "./artifacts/triage-lora"

print("✅ 评估环境初始化完成")


✅ 评估环境初始化完成


### 2.加载训练好的 LoRA 分诊模型

In [3]:
def load_triage_lora_model():
    """加载底座模型 + 挂载LoRA适配器，固定eval推理模式"""
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(OUT_DIR)
    
    # 重建原始分类模型结构
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=len(LABELS)
    )
    
    # 挂载训练好的LoRA权重
    lora_model = PeftModel.from_pretrained(base_model, OUT_DIR)
    
    # 关闭dropout、梯度，纯推理模式
    lora_model.eval()
    
    return tokenizer, lora_model

tokenizer, model = load_triage_lora_model()
print("✅ LoRA 分诊模型加载成功，进入推理模式")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ LoRA 分诊模型加载成功，进入推理模式


### 3. 批量推理函数（无梯度加速）

@torch.no_grad() —— 装饰器，修饰整个函数

with torch.no_grad(): 上下文管理器，修饰一小块代码

In [4]:
@torch.no_grad()
def batch_predict(text_list: list[str]) -> list[int]:
    """批量对症状文本做分诊预测，返回类别ID"""
    inputs = tokenizer(
        text_list,
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt"
    )
    logits = model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1).tolist()
    return pred_ids

print("✅ 批量预测函数定义完成")


✅ 批量预测函数定义完成


### 4. 加载测试集（ unseen 全新句式）

In [5]:
# 读取独立测试集（疑问句模板，训练完全没见过）
test_path = Path("finetuning/data/test.jsonl")
test_rows = [
    json.loads(line)
    for line in test_path.read_text().splitlines()
    if line.strip()
]

# 构建标签映射
label2id = {label: idx for idx, label in enumerate(LABELS)}

# 提取测试集文本 & 真实标签
test_texts = [row["text"] for row in test_rows]
y_true = [label2id[row["label"]] for row in test_rows]

print(f"✅ 测试集加载完成，样本总数：{len(test_texts)}")
print("样本句式全部为【未知疑问句】，可真实评估泛化能力")


✅ 测试集加载完成，样本总数：305
样本句式全部为【未知疑问句】，可真实评估泛化能力


### 5. 全量测试集推理 + 指标计算    

compute_metrics：只返回整体准确率、整体 F1，供 Trainer 训练期间打分；

classification_report：输出每一类标签独立的评估数据，可以查看模型是不是分不清某两种症状等级

In [6]:
# 批量预测全部测试集
y_pred = batch_predict(test_texts)

# 输出精细分类报告
print("=" * 60)
print("📊 医疗分诊模型 分类评估报告（ Held-Out 测试集 ）")
print("=" * 60)
print(classification_report(
    y_true,
    y_pred,
    target_names=LABELS,
    zero_division=0
))


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


📊 医疗分诊模型 分类评估报告（ Held-Out 测试集 ）
              precision    recall  f1-score   support

   emergency       1.00      0.94      0.97        85
      urgent       0.94      0.91      0.92        65
     routine       0.90      1.00      0.95        80
   self_care       0.97      0.95      0.96        75

    accuracy                           0.95       305
   macro avg       0.95      0.95      0.95       305
weighted avg       0.95      0.95      0.95       305



### 6. 输出混淆矩阵（核心面试亮点）

行：真实标准答案

列：模型给出的预测结果

In [7]:
print("=" * 60)
print("📈 混淆矩阵（行=真实标签 | 列=预测标签）")
print("类别顺序：", LABELS)
print("=" * 60)

cm = confusion_matrix(y_true, y_pred)
print(cm)

print("\n💡 模型容错特性分析：")
print("医疗场景最优容错：急症不被漏判，宁可普通症状判偏紧急")
print("可观察：模型错误集中在保守预判，无高危漏诊")


📈 混淆矩阵（行=真实标签 | 列=预测标签）
类别顺序： ['emergency', 'urgent', 'routine', 'self_care']
[[80  4  1  0]
 [ 0 59  4  2]
 [ 0  0 80  0]
 [ 0  0  4 71]]

💡 模型容错特性分析：
医疗场景最优容错：急症不被漏判，宁可普通症状判偏紧急
可观察：模型错误集中在保守预判，无高危漏诊


### 7. 自定义单样本测试（手动验证）

In [8]:
def triage_infer(text: str):
    """单条症状分诊推理"""
    pred_id = batch_predict([text])[0]
    return LABELS[pred_id]

# 医疗典型测试用例
case_list = [
    "I have sudden severe chest pain and difficulty breathing",
    "I have a mild headache every month before my period",
    "I have a runny nose and mild sore throat",
    "I have a high fever lasting three days"
]

print("🧪 手动样例推理测试：\n")
for case in case_list:
    res = triage_infer(case)
    print(f"症状：{case}")
    print(f"分诊结果：{res}\n")


🧪 手动样例推理测试：

症状：I have sudden severe chest pain and difficulty breathing
分诊结果：emergency

症状：I have a mild headache every month before my period
分诊结果：routine

症状：I have a runny nose and mild sore throat
分诊结果：self_care

症状：I have a high fever lasting three days
分诊结果：urgent

